# Bakeries Data Layer – Transformation & Preprocessing

# Objective 
    Prepare a unified, validated bakery dataset ready for integration into the platform database.

# Libraries

In [7]:
# Install Libraries

! pip install osmnx geopandas pandas

In [8]:
# Import Libraries
import pandas as pd
import geopandas as gpd # to work with geospatial data
import osmnx as ox # to fetch data from OpenStreetMap
from shapely.geometry import Point
import json
import requests
import numpy as np

# 1. Data Extraction – OpenStreetMap (Primary Source)

 ##  Bakery Definition

Establishments primarily selling bread and pastries.

 ## Eligible OSM Tags

 - shop=bakery

 - shop=pastry

- bakery=yes

In [9]:
ox.settings.log_console = True
ox.settings.use_cache = True


osm_tags = {
"shop": ["bakery", "pastry"],
"bakery": "yes"
}


bakeries_raw = ox.features_from_place(
"Berlin, Germany",
tags=osm_tags
).reset_index()

#len(bakeries_raw)
bakeries_raw.head()

,element,id,geometry,addr:city,addr:country,addr:housenumber,addr:postcode,addr:street,addr:suburb,shop,...,room,roof:levels,office,building:part,covered,nohousenumber,coffee,location,shop_1,description:de
0,node,58489996,POINT (13.40829 52.51035),Berlin,DE,77,10179,Alte Jakobstraße,Mitte,bakery,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,253999302,POINT (13.35618 52.48628),Berlin,DE,22,10827,Hauptstraße,Schöneberg,bakery,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,254333065,POINT (13.32023 52.49613),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,254992626,POINT (13.26299 52.48796),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,258050855,POINT (13.29729 52.50696),Berlin,DE,80,10627,Kantstraße,Charlottenburg,bakery,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
excluded_amenities = ["cafe", "restaurant", "fast_food", "supermarket"]

if "amenity" in bakeries_raw.head().columns:
    bakeries_raw = bakeries_raw[
        ~bakeries_raw["amenity"].isin(excluded_amenities)
    ]


In [12]:
bakeries_raw.columns

Index(['element', 'id', 'geometry', 'addr:city', 'addr:country',
       'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:suburb',
       'shop',
       ...
       'room', 'roof:levels', 'office', 'building:part', 'covered',
       'nohousenumber', 'coffee', 'location', 'shop_1', 'description:de'],
      dtype='object', length=204)

# 2. Initial Inspection

In [13]:
bakeries_raw.head()
#Validate tagging strategy and detect overlaps with cafes or restaurants.
bakeries_raw[["name", "shop", "bakery", "amenity"]].sample(10)

,name,shop,bakery,amenity
1309,Junge,bakery,NaN,NaN
711,Choco Latte,bakery,NaN,NaN
562,denns Biomarkt - Backshop,bakery,NaN,NaN
493,Wahl,bakery,NaN,NaN
236,Bäcker Walf,bakery,NaN,NaN
270,Oder Brot,bakery,NaN,NaN
1021,Familienbäckerei Rösler,bakery,NaN,NaN
1300,Steinecke,bakery,NaN,NaN
881,Ma Petite,pastry,NaN,NaN
830,c'est la vie,bakery,NaN,NaN


Inspection of OSM bakery tags confirms that Berlin bakeries are predominantly classified as  shop=bakery, with a smaller but relevant subset tagged as shop=pastry (e.g., Konditoreien and pastry-focused shops). Both tags are therefore included in the unified Bakeries dataset.

# 3. Column Cleaning & Standardization
## Rules

- All columns lowercase

- Snake_case naming

- Unified address fields#

In [14]:
bakeries_raw.columns = [
col.lower().replace(":", "_") for col in bakeries_raw.columns
]


rename_map = {
"addr_street": "street",
"addr_housenumber": "house_number",
"addr_postcode": "postal_code"
}


bakeries = bakeries_raw.rename(columns=rename_map)

# 4. Exclude Overlapping POI Categories

To avoid duplication with other POI layers (Cafes, Supermarkets), we exclude bakeries primarily tagged as:

- cafe

- restaurant

- fast_food

In [15]:
excluded_amenities = ["cafe", "restaurant", "fast_food", "supermarket"]


if "amenity" in bakeries.columns:
 bakeries = bakeries[~bakeries["amenity"].isin(excluded_amenities)]


len(bakeries)

1309

# 5. Bakery Type & Chain Classification
## Logic

 - Presence of brand or operator → chain bakery

 - Otherwise → artisanal bakery

In [16]:
def classify_bakery(row):
 if pd.notna(row.get("brand")) or pd.notna(row.get("operator")):
  return "chain"
 return "artisanal"


bakeries["bakery_type"] = bakeries.apply(classify_bakery, axis=1)
bakeries["is_chain"] = bakeries["bakery_type"] == "chain"


bakeries["brand_name"] = bakeries["brand"].fillna(bakeries.get("operator"))

# 6.Geospatial Validation 
 

## 6.1 CRS & Geometry Validity

In [21]:
bakeries = bakeries.set_crs(epsg=4326, allow_override=True)

# Remove rows with missing geometry
bakeries = bakeries[bakeries.geometry.notnull()]

# Remove invalid geometries
bakeries = bakeries[bakeries.geometry.is_valid]


In [22]:
bakery_gdf = gpd.GeoDataFrame(
    bakeries,
    geometry=gpd.points_from_xy(bakeries.longitude, bakeries.latitude),
    crs="EPSG:4326"
)


In [25]:
bakery_gdf.columns

Index(['element', 'id', 'geometry', 'addr_city', 'addr_country',
       'house_number', 'postal_code', 'street', 'addr_suburb', 'shop',
       ...
       'nohousenumber', 'coffee', 'location', 'shop_1', 'description_de',
       'bakery_type', 'is_chain', 'brand_name', 'latitude', 'longitude'],
      dtype='object', length=209)

In [ ]:
# Load official Berlin districts GeoDataFrame from lor_ortsteile.geojson
berlin_districts_gdf = gpd.read_file("/Users/nigar_kauser/Documents/Webeet_internship/Bakeries/layered-populate-data-pool-da/mapping/lor_ortsteile.geojson")
berlin_districts_gdf = berlin_districts_gdf.to_crs(epsg=4326)


In [27]:
berlin_districts_gdf.columns

Index(['gml_id', 'spatial_name', 'spatial_alias', 'spatial_type', 'OTEIL',
       'BEZIRK', 'FLAECHE_HA', 'geometry'],
      dtype='object')

In [28]:
# Spatial join: matching your bakeries with district and neighborhoods
bakery_with_districts = gpd.sjoin(
    bakery_gdf,
    berlin_districts_gdf[["BEZIRK", "spatial_name", "OTEIL","geometry"]],
    how="left",
    predicate="within"
)
bakery_with_districts.head()

,element,id,geometry,addr_city,addr_country,house_number,postal_code,street,addr_suburb,shop,...,description_de,bakery_type,is_chain,brand_name,latitude,longitude,index_right,BEZIRK,spatial_name,OTEIL
0,node,58489996,POINT (13.40829 52.51035),Berlin,DE,77,10179,Alte Jakobstraße,Mitte,bakery,...,NaN,artisanal,False,NaN,52.510346,13.408290,0,Mitte,0101,Mitte
1,node,253999302,POINT (13.35618 52.48628),Berlin,DE,22,10827,Hauptstraße,Schöneberg,bakery,...,NaN,chain,True,Back-Factory,52.486277,13.356180,44,Tempelhof-Schöneberg,0701,Schöneberg
2,node,254333065,POINT (13.32023 52.49613),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,artisanal,False,NaN,52.496132,13.320234,22,Charlottenburg-Wilmersdorf,0402,Wilmersdorf
3,node,254992626,POINT (13.26299 52.48796),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,chain,True,Steinecke,52.487959,13.262985,24,Charlottenburg-Wilmersdorf,0404,Grunewald
4,node,258050855,POINT (13.29729 52.50696),Berlin,DE,80,10627,Kantstraße,Charlottenburg,bakery,...,NaN,artisanal,False,NaN,52.506957,13.297293,21,Charlottenburg-Wilmersdorf,0401,Charlottenburg


In [29]:
##just renaming columns for proper schema
bakery_with_districts = bakery_with_districts.rename(columns={
    "BEZIRK": "district",
    "OTEIL": "neighborhood",
    "spatial_name": "neighborhood_id"
}).drop(columns=["index_right"])  # drop district_number if not needed
bakery_with_districts.head()

,element,id,geometry,addr_city,addr_country,house_number,postal_code,street,addr_suburb,shop,...,shop_1,description_de,bakery_type,is_chain,brand_name,latitude,longitude,district,neighborhood_id,neighborhood
0,node,58489996,POINT (13.40829 52.51035),Berlin,DE,77,10179,Alte Jakobstraße,Mitte,bakery,...,NaN,NaN,artisanal,False,NaN,52.510346,13.408290,Mitte,0101,Mitte
1,node,253999302,POINT (13.35618 52.48628),Berlin,DE,22,10827,Hauptstraße,Schöneberg,bakery,...,NaN,NaN,chain,True,Back-Factory,52.486277,13.356180,Tempelhof-Schöneberg,0701,Schöneberg
2,node,254333065,POINT (13.32023 52.49613),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,NaN,artisanal,False,NaN,52.496132,13.320234,Charlottenburg-Wilmersdorf,0402,Wilmersdorf
3,node,254992626,POINT (13.26299 52.48796),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,NaN,chain,True,Steinecke,52.487959,13.262985,Charlottenburg-Wilmersdorf,0404,Grunewald
4,node,258050855,POINT (13.29729 52.50696),Berlin,DE,80,10627,Kantstraße,Charlottenburg,bakery,...,NaN,NaN,artisanal,False,NaN,52.506957,13.297293,Charlottenburg-Wilmersdorf,0401,Charlottenburg


In [30]:
# District mapping (official codes as strings)
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

# Apply mapping to create district_id column (string)
bakery_with_districts['district_id'] = bakery_with_districts['district'].map(district_mapping).astype(str)

# (Optional) Check if some districts were not mapped
#unmapped = df[~df['district'].isin(district_mapping.keys())]['district'].unique()
#if len(unmapped) > 0:
    #print("⚠️ Unmapped districts found:", unmapped)
bakery_with_districts.head()

,element,id,geometry,addr_city,addr_country,house_number,postal_code,street,addr_suburb,shop,...,description_de,bakery_type,is_chain,brand_name,latitude,longitude,district,neighborhood_id,neighborhood,district_id
0,node,58489996,POINT (13.40829 52.51035),Berlin,DE,77,10179,Alte Jakobstraße,Mitte,bakery,...,NaN,artisanal,False,NaN,52.510346,13.408290,Mitte,0101,Mitte,11001001
1,node,253999302,POINT (13.35618 52.48628),Berlin,DE,22,10827,Hauptstraße,Schöneberg,bakery,...,NaN,chain,True,Back-Factory,52.486277,13.356180,Tempelhof-Schöneberg,0701,Schöneberg,11007007
2,node,254333065,POINT (13.32023 52.49613),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,artisanal,False,NaN,52.496132,13.320234,Charlottenburg-Wilmersdorf,0402,Wilmersdorf,11004004
3,node,254992626,POINT (13.26299 52.48796),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,chain,True,Steinecke,52.487959,13.262985,Charlottenburg-Wilmersdorf,0404,Grunewald,11004004
4,node,258050855,POINT (13.29729 52.50696),Berlin,DE,80,10627,Kantstraße,Charlottenburg,bakery,...,NaN,artisanal,False,NaN,52.506957,13.297293,Charlottenburg-Wilmersdorf,0401,Charlottenburg,11004004


In [31]:
bakery_with_districts[
    ["name", "district", "district_id", "neighborhood", "neighborhood_id"]
].head()

,name,district,district_id,neighborhood,neighborhood_id
0,NaN,Mitte,11001001,Mitte,0101
1,Back-Factory,Tempelhof-Schöneberg,11007007,Schöneberg,0701
2,Brot & Brötchen,Charlottenburg-Wilmersdorf,11004004,Wilmersdorf,0402
3,Steinecke,Charlottenburg-Wilmersdorf,11004004,Grunewald,0404
4,Wilmina Brot,Charlottenburg-Wilmersdorf,11004004,Charlottenburg,0401


In [ ]:
bakery_with_districts.head()

,element,id,geometry,addr_city,addr_country,house_number,postal_code,street,addr_suburb,shop,...,latitude,longitude,index_right,gml_id,spatial_name,spatial_alias,spatial_type,OTEIL,BEZIRK,FLAECHE_HA
0,node,58489996,POINT (13.40829 52.51035),Berlin,DE,77,10179,Alte Jakobstraße,Mitte,bakery,...,52.510346,13.408290,0,re_ortsteil.0101,0101,Mitte,Polygon,Mitte,Mitte,1063.8748
1,node,253999302,POINT (13.35618 52.48628),Berlin,DE,22,10827,Hauptstraße,Schöneberg,bakery,...,52.486277,13.356180,44,re_ortsteil.0701,0701,Schöneberg,Polygon,Schöneberg,Tempelhof-Schöneberg,1060.1196
2,node,254333065,POINT (13.32023 52.49613),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,52.496132,13.320234,22,re_ortsteil.0402,0402,Wilmersdorf,Polygon,Wilmersdorf,Charlottenburg-Wilmersdorf,713.3992
3,node,254992626,POINT (13.26299 52.48796),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,52.487959,13.262985,24,re_ortsteil.0404,0404,Grunewald,Polygon,Grunewald,Charlottenburg-Wilmersdorf,2244.1743
4,node,258050855,POINT (13.29729 52.50696),Berlin,DE,80,10627,Kantstraße,Charlottenburg,bakery,...,52.506957,13.297293,21,re_ortsteil.0401,0401,Charlottenburg,Polygon,Charlottenburg,Charlottenburg-Wilmersdorf,1042.1161


In [32]:
[col for col in bakery_with_districts.columns if "osm" in col.lower()]
bakery_with_districts.columns


Index(['element', 'id', 'geometry', 'addr_city', 'addr_country',
       'house_number', 'postal_code', 'street', 'addr_suburb', 'shop',
       ...
       'description_de', 'bakery_type', 'is_chain', 'brand_name', 'latitude',
       'longitude', 'district', 'neighborhood_id', 'neighborhood',
       'district_id'],
      dtype='object', length=213)

# 7. District & Neighborhood Enrichment

Each bakery must be mapped to:

 - District

 - Neighborhood

 - District ID

 - Neighborhood ID

# 8. Final Unified Schema Selection

In [33]:
final_columns = [
"name",
"latitude",
"longitude",
"geometry",
"district",
"neighborhood",
"district_id",
"neighborhood_id",
"bakery_type",
"is_chain",
"brand_name",
"street",
"house_number",
"postal_code",
"opening_hours",
"website"
]
final_bakeries = bakery_with_districts[final_columns].copy()
final_bakeries.head()

,name,latitude,longitude,geometry,district,neighborhood,district_id,neighborhood_id,bakery_type,is_chain,brand_name,street,house_number,postal_code,postal_code,opening_hours,website
0,NaN,52.510346,13.408290,POINT (13.40829 52.51035),Mitte,Mitte,11001001,0101,artisanal,False,NaN,Alte Jakobstraße,77,10179,NaN,NaN,NaN
1,Back-Factory,52.486277,13.356180,POINT (13.35618 52.48628),Tempelhof-Schöneberg,Schöneberg,11007007,0701,chain,True,Back-Factory,Hauptstraße,22,10827,NaN,Mo-Fr 06:00-19:00; Sa 06:30-18:00; Su 07:00-16:00,NaN
2,Brot & Brötchen,52.496132,13.320234,POINT (13.32023 52.49613),Charlottenburg-Wilmersdorf,Wilmersdorf,11004004,0402,artisanal,False,NaN,NaN,NaN,NaN,NaN,Mo-Fr 07:00-17:00; Sa 07:00-14:00; Su 07:00-14:00,NaN
3,Steinecke,52.487959,13.262985,POINT (13.26299 52.48796),Charlottenburg-Wilmersdorf,Grunewald,11004004,0404,chain,True,Steinecke,NaN,NaN,NaN,NaN,Mo-Fr 06:30-18:00; Sa 07:00-14:00; Su 07:00-12:00,NaN
4,Wilmina Brot,52.506957,13.297293,POINT (13.29729 52.50696),Charlottenburg-Wilmersdorf,Charlottenburg,11004004,0401,artisanal,False,NaN,Kantstraße,80,10627,NaN,We-Su 08:00-16:00,NaN


# 9. Quality Assurance Checks

In [34]:
qa = {
"total_records": len(final_bakeries),
"missing_names": final_bakeries["name"].isna().sum(),
"missing_coords": final_bakeries[
final_bakeries["latitude"].isna() | final_bakeries["longitude"].isna()
].shape[0],
"chain_share_percent": round(final_bakeries["is_chain"].mean() * 100, 2)
}


qa

{'total_records': 1309,
 'missing_names': np.int64(21),
 'missing_coords': 0,
 'chain_share_percent': np.float64(25.06)}

In [38]:
bakery_with_districts.sample(5, random_state=42)

,element,id,geometry,addr_city,addr_country,house_number,postal_code,street,addr_suburb,shop,...,bakery_type,is_chain,brand_name,latitude,longitude,district,neighborhood_id,neighborhood,district_id,source_ids
1202,node,11583913969,POINT (13.4229 52.48855),Berlin,NaN,70a,10967,Urbanstraße,NaN,bakery,...,artisanal,False,NaN,52.488551,13.422899,Friedrichshain-Kreuzberg,0202,Kreuzberg,11002002,node/11583913969
1099,node,9492136446,POINT (13.453 52.54857),Berlin,DE,50,13088,Berliner Allee,Weißensee,bakery,...,chain,True,BackWerk,52.548569,13.452997,Pankow,0302,Weißensee,11003003,node/9492136446
1029,node,7761740685,POINT (13.34529 52.48067),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,artisanal,False,NaN,52.480671,13.345287,Tempelhof-Schöneberg,0701,Schöneberg,11007007,node/7761740685
852,node,5324729970,POINT (13.44793 52.47327),Berlin,NaN,10,12055,Zwiestädter Straße,NaN,bakery,...,artisanal,False,NaN,52.473273,13.447928,Neukölln,0801,Neukölln,11008008,node/5324729970
1251,node,12512754450,POINT (13.43258 52.46745),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,artisanal,False,NaN,52.467448,13.432581,Neukölln,0801,Neukölln,11008008,node/12512754450


In [42]:
bakery_with_districts["source_ids"] = (
    bakery_with_districts["element"].astype(str)
    + "/"
    + bakery_with_districts["id"].astype(str)
)


# 10. Data Source Attribution

In [36]:
final_bakeries["data_source"] = "OSM"
final_bakeries["source_ids"] = bakery_with_districts["source_ids"]

In [37]:
final_bakeries[["name", "data_source", "source_ids"]].head()


,name,data_source,source_ids
0,NaN,OSM,node/58489996
1,Back-Factory,OSM,node/253999302
2,Brot & Brötchen,OSM,node/254333065
3,Steinecke,OSM,node/254992626
4,Wilmina Brot,OSM,node/258050855


In [ ]:
final_bakeries.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 1309 entries, 0 to 1366
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   name             1288 non-null   object  
 1   latitude         1309 non-null   float64 
 2   longitude        1309 non-null   float64 
 3   geometry         1309 non-null   geometry
 4   district         1309 non-null   object  
 5   neighborhood     1309 non-null   object  
 6   district_id      1309 non-null   object  
 7   neighborhood_id  1309 non-null   object  
 8   bakery_type      1309 non-null   object  
 9   is_chain         1309 non-null   bool    
 10  brand_name       328 non-null    object  
 11  street           715 non-null    object  
 12  house_number     689 non-null    object  
 13  postal_code      659 non-null    object  
 14  postal_code      1 non-null      object  
 15  opening_hours    922 non-null    object  
 16  website          191 non-null    object

In [ ]:
final_bakeries.columns[final_bakeries.columns.duplicated()]


Index([], dtype='object')

In [ ]:
# Keep only the first occurrence of each column, drop duplicates
final_bakeries = final_bakeries.loc[:, ~final_bakeries.columns.duplicated()]

In [45]:
print("✅ Dataset after Steps A - D cleaning and transforming\n")

# Shape of dataframe
print(f"Number of rows: {final_bakeries.shape[0]}")
print(f"Number of columns: {final_bakeries.shape[1]}")

# Column list
print("\nRemaining columns:")
print(final_bakeries.columns.tolist())

# Missing values check
missing = final_bakeries.isnull().sum()
print("\nMissing values after cleaning and transforming :")
print(missing)

✅ Dataset after Steps A - D cleaning and transforming

Number of rows: 1309
Number of columns: 18

Remaining columns:
['name', 'latitude', 'longitude', 'geometry', 'district', 'neighborhood', 'district_id', 'neighborhood_id', 'bakery_type', 'is_chain', 'brand_name', 'street', 'house_number', 'postal_code', 'opening_hours', 'website', 'data_source', 'source_ids']

Missing values after cleaning and transforming :
name                 21
latitude              0
longitude             0
geometry              0
district              0
neighborhood          0
district_id           0
neighborhood_id       0
bakery_type           0
is_chain              0
brand_name          981
street              593
house_number        619
postal_code         649
opening_hours       387
website            1118
data_source           0
source_ids            0
dtype: int64


In [44]:

# 1. Fix Duplicate Columns (Keeping the one with more data)
# This identifies columns with the same name and drops the one that is mostly empty
cols = pd.Series(final_bakeries.columns)
for dup in cols[cols.duplicated()].unique():
    # Keep the first, drop the rest
    cols_to_drop = [i for i, v in enumerate(final_bakeries.columns) if v == dup][1:]
    final_bakeries = final_bakeries.iloc[:, ~final_bakeries.columns.duplicated()]

# 11. Save Processed Output

In [49]:

# Save final bakeries dataset as CSV (non-spatial version)
final_bakeries.drop(columns=["geometry"], errors="ignore").to_csv(
    "../sources/bakeries_berlin.csv",
    index=False
)
# Save final bakeries dataset as GeoJSON (spatial version)
final_bakeries.to_file(
    "../sources/bakeries_berlin.geojson",
    driver="GeoJSON"
)